# Task 3: Partition Pruning Experiment (Queries 1, 6)


## Connect to Spark

In [ ]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import (
    DEFAULT_INTEGRATED_TABLE_PATH,
    INTEGRATED_TABLE_VIEW,
    execute_registered_query,
    register_integrated_taxi_trips,
)

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-PartitionPruning-Query1-6')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 17:02:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Define the two layouts

- February is used because the data covers January-March 2024. 
- Partition prunnig by skipping January and March.

In [ ]:
PARTITIONED_TABLE_PATH = DEFAULT_INTEGRATED_TABLE_PATH
UNPARTITIONED_TABLE_PATH = (
    's3a://delta-data/gold/gold_integrated_taxi_trips_unpartitioned'
)
QUERY_NAMES = (
    'query_01_monthly_zone_demand',
    'query_06_monthly_taxi_demand_trend',
)
REPETITIONS = 3

## Create the unpartitioned layout

In [ ]:
partitioned_table = spark.read.format('delta').load(PARTITIONED_TABLE_PATH)

(
    partitioned_table.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(UNPARTITIONED_TABLE_PATH)
)

unpartitioned_table = (
    spark.read.format('delta').load(UNPARTITIONED_TABLE_PATH)
)

assert partitioned_table.count() == unpartitioned_table.count()
print('Unpartitioned layout created with the same number of rows.')

26/09/17 17:02:03 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/17 17:02:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Unpartitioned layout created with the same number of rows.


## Measure both layouts

Each layout receives one warm-up followed by three timed runs.

In [ ]:
def measure_layout(table_path):
    spark.catalog.clearCache()
    register_integrated_taxi_trips(spark, table_path)
    return {
        query_name: measure_registered_query(spark, query_name, REPETITIONS)
        for query_name in QUERY_NAMES
    }

partitioned_results = measure_layout(PARTITIONED_TABLE_PATH)
unpartitioned_results = measure_layout(UNPARTITIONED_TABLE_PATH)

for query_name in QUERY_NAMES:
    assert rows_match(
        partitioned_results[query_name]['rows'],
        unpartitioned_results[query_name]['rows'],
    ), f'Partitioned and unpartitioned results differ for {query_name}.'

print('Both layouts return the same results within float tolerance.')

Both layouts return the same results within float tolerance.


## Partitioned physical plans

- `PartitionFilters` contains the February 2024 range.

In [ ]:
register_integrated_taxi_trips(spark, PARTITIONED_TABLE_PATH)
for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_03_air_quality_vs_demand_february_2024 ---
== Physical Plan ==
AdaptiveSparkPlan (20)
+- Sort (19)
   +- Exchange (18)
      +- BroadcastNestedLoopJoin Cross BuildRight (17)
         :- HashAggregate (6)
         :  +- Exchange (5)
         :     +- HashAggregate (4)
         :        +- Project (3)
         :           +- Filter (2)
         :              +- Scan parquet  (1)
         +- BroadcastExchange (16)
            +- HashAggregate (15)
               +- Exchange (14)
                  +- HashAggregate (13)
                     +- HashAggregate (12)
                        +- Exchange (11)
                           +- HashAggregate (10)
                              +- Project (9)
                                 +- Filter (8)
                                    +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [pu_datetime_hour#7313, average_value_pu#7330, date#7344]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trip

## Unpartitioned physical plans

- It does not contain a `PartitionFilters` entry.

In [ ]:
register_integrated_taxi_trips(spark, UNPARTITIONED_TABLE_PATH)
for query_name in QUERY_NAMES:
    print(f'\n--- {query_name} ---')
    execute_registered_query(spark, query_name).explain('formatted')


--- query_03_air_quality_vs_demand_february_2024 ---
== Physical Plan ==
AdaptiveSparkPlan (20)
+- Sort (19)
   +- Exchange (18)
      +- BroadcastNestedLoopJoin Cross BuildRight (17)
         :- HashAggregate (6)
         :  +- Exchange (5)
         :     +- HashAggregate (4)
         :        +- Project (3)
         :           +- Filter (2)
         :              +- Scan parquet  (1)
         +- BroadcastExchange (16)
            +- HashAggregate (15)
               +- Exchange (14)
                  +- HashAggregate (13)
                     +- HashAggregate (12)
                        +- Exchange (11)
                           +- HashAggregate (10)
                              +- Project (9)
                                 +- Filter (8)
                                    +- Scan parquet  (7)


(1) Scan parquet 
Output [3]: [pu_datetime_hour#7872, average_value_pu#7889, date#7903]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/gold/gold_integrated_taxi_trip

## Benchmark summary

In [ ]:
summary = []
for query_name in QUERY_NAMES:
    partitioned = partitioned_results[query_name]
    unpartitioned = unpartitioned_results[query_name]
    summary.append({
        'query': query_name,
        'partitioned_samples_seconds': [
            round(value, 3) for value in partitioned['samples_seconds']
        ],
        'partitioned_median_seconds': round(partitioned['median_seconds'], 3),
        'unpartitioned_samples_seconds': [
            round(value, 3) for value in unpartitioned['samples_seconds']
        ],
        'unpartitioned_median_seconds': round(unpartitioned['median_seconds'], 3),
        'partitioned_speedup_factor': round(
            unpartitioned['median_seconds'] / partitioned['median_seconds'],
            2,
        ),
    })
summary

[{'query': 'query_03_air_quality_vs_demand_february_2024',
  'partitioned_samples_seconds': [0.532, 0.509, 0.441],
  'partitioned_median_seconds': 0.509,
  'unpartitioned_samples_seconds': [0.388, 0.288, 0.354],
  'unpartitioned_median_seconds': 0.354,
  'partitioned_speedup_factor': 0.7},
 {'query': 'query_05_peak_travel_hours_february_2024',
  'partitioned_samples_seconds': [0.476, 0.379, 0.36],
  'partitioned_median_seconds': 0.379,
  'unpartitioned_samples_seconds': [0.387, 0.345, 0.334],
  'unpartitioned_median_seconds': 0.345,
  'partitioned_speedup_factor': 0.91}]

## Cleanup

In [ ]:
spark.catalog.clearCache()
spark.stop()